# 5'UTR Marker Analysis: NT96 / NT192 / NT384 / human / human_truncated / random

This notebook tests four sequence-derived markers across six sequence groups, reusing the
analysis functions already written in this repository (`src/rna_analysis.py`, `src/constants.py`,
`src/KLD_calculation.py`, `src/rng_sequences.py`):

1. **Nucleotide probabilities** — per-base (A/C/G/U) frequencies and GC content.
2. **uORFs in reading frame** — frame-resolved upstream ORF counts/lengths for 5 (near-)cognate start codons.
3. **Secondary structure / MFE** — sliding-window minimum free energy via ViennaRNA.
4. **Markov (dinucleotide transition) probabilities** — first-order transition matrices + pairwise JSD between groups.

Each marker below follows the same three-step pattern:
**1) write the function (only where one doesn't already exist in `src/`), 2) apply it across all six
groups, 3) plot the result** — with comments explaining each step.

**Data sources:**
- `data/new_dataset.csv` already has an explicit `Group` column and supplies the `human` and `random` groups.
- `data/delivered_twist_pool.csv` has no `Group` column at all — the `human_truncated` sequences are
  mixed in with everything else in the pool and only identifiable by their `transcript_id`: rows
  starting with `"ENST"` (an Ensembl transcript ID) are `human_truncated`; everything else in that
  file is a different design and gets filtered out as a separate step.
- `NT96` / `NT192` / `NT384` don't have a real source wired up yet (TODO, flagged again below) — synthetic
  placeholder sequences stand in for now purely so the rest of the notebook runs end-to-end.

If either real CSV is missing, a small synthetic stand-in is generated instead (using this repo's own
`src.rng_sequences.randomseqs`) so the notebook still runs — replace the relevant block once the real
file is available.


## 0. Setup

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")   # matches the convention used in model.ipynb / test.ipynb
sys.path.append(".")

import src.constants as constants
import src.rna_analysis as rna
import src.KLD_calculation as kldcalc
import src.rng_sequences as rngseq

sns.set_style("whitegrid")
plt.rc("font", size=12)

# Marker 3 (MFE) needs ViennaRNA - check it's importable up front so we can skip that section cleanly if not.
try:
    import ViennaRNA as vrna
    MFE_AVAILABLE = True
except ImportError:
    MFE_AVAILABLE = False
    warnings.warn("ViennaRNA is not installed - marker 3 (MFE) will be skipped. "
                   "Install it with: pip install ViennaRNA")
print("Setup complete. MFE/ViennaRNA available:", MFE_AVAILABLE)


## 1. Configuration

Paths, column names, and the ID rule used to pull `human_truncated` out of the twist pool file.
Adjust these if your real column names differ.

In [ ]:
# --- File locations -------------------------------------------------------
# new_dataset.csv          -> 'human' and 'random' groups (explicit Group column)
# delivered_twist_pool.csv -> 'human_truncated', identified by ENST prefix on transcript_id
# samples_by_library.csv   -> NT96 / NT192 / NT384, identified by a 'library' column
NEW_DATASET_PATH     = "data/new_dataset.csv"
TWIST_POOL_PATH      = "data/delivered_twist_pool.csv"
SAMPLES_BY_LIB_PATH  = "data/samples_by_library.csv"

# --- Column names -----------------------------------------------------------
SEQ_COL   = "Sequence"  # sequence column shared across all three files
GROUP_COL = "Group"     # unified group label column used everywhere downstream

# Twist pool specific
TWIST_SEQ_COL    = "Sequence"       # sequence column inside delivered_twist_pool.csv
TWIST_ID_COL     = "transcript_id"  # identifier column inside delivered_twist_pool.csv
TWIST_ID_PREFIX  = "ENST"           # rows whose ID starts with this are the human_truncated set
TWIST_GROUP_NAME = "human_truncated"

# Samples-by-library specific
NT_LIB_COL  = "library"                    # group label column inside samples_by_library.csv
NT_GROUPS   = ["NT96", "NT192", "NT384"]   # which library values to keep

GROUPS       = ["NT96", "NT192", "NT384", "human", "human_truncated", "random"]
START_CODONS = ["AUG", "CUG", "GUG", "UUG", "ACG"]   # used for the uORF marker

MFE_SAMPLE_SIZE = 150    # ViennaRNA folding is slow; subsample large groups for marker 3.
                         # Set to None to fold every sequence in every group.
RANDOM_SEED = 42

PALETTE = dict(zip(GROUPS, sns.color_palette("Set2", n_colors=len(GROUPS))))


## 2. Placeholder data helpers

These only run if a real file is missing, or for `NT96`/`NT192`/`NT384` (no real source specified yet).
They exist purely so the notebook is runnable end-to-end right now — safe to delete once everything
is wired up to real data.

In [ ]:
def _make_demo_new_dataset(seed=RANDOM_SEED):
    """Stand-in for new_dataset.csv, used only if that file is not found on disk."""
    human_  = rngseq.randomseqs(60, 150, "human",  gc_content=0.62, seed=seed)
    random_ = rngseq.randomseqs(60, 150, "random", gc_content=0.50, seed=seed + 1)
    return pd.DataFrame(np.vstack([human_, random_]), columns=[SEQ_COL, GROUP_COL])


def _make_demo_twist_pool(seed=RANDOM_SEED, n_enst=60, n_other=40):
    """Stand-in for delivered_twist_pool.csv, used only if that file is not found on disk.
    Mixes ENST-prefixed rows (the real human_truncated target) with other IDs so the
    ENST-filtering step below has something realistic to exercise."""
    rng = np.random.default_rng(seed)
    enst_ids  = [f"ENST{rng.integers(10**10, 10**11 - 1)}" for _ in range(n_enst)]
    other_ids = [f"CTRL_{i:04d}" for i in range(n_other)]
    enst_seqs  = rngseq.randomseqs(n_enst,  96, "tmp", gc_content=0.62, seed=seed + 2)[:, 0]
    other_seqs = rngseq.randomseqs(n_other, 96, "tmp", gc_content=0.50, seed=seed + 3)[:, 0]
    return pd.DataFrame({
        TWIST_ID_COL:  enst_ids + other_ids,
        TWIST_SEQ_COL: list(enst_seqs) + list(other_seqs),
    })


def _make_demo_samples_by_library(seed=RANDOM_SEED):
    """Stand-in for samples_by_library.csv, used only if that file is not found on disk."""
    specs = [("NT96", 96), ("NT192", 192), ("NT384", 384)]
    frames = []
    for i, (name, length) in enumerate(specs):
        arr = rngseq.randomseqs(60, length, name, gc_content=0.55, seed=seed + 10 + i)
        frames.append(pd.DataFrame(arr, columns=[SEQ_COL, NT_LIB_COL]))
    return pd.concat(frames, ignore_index=True)


## 3. Load sequence data

### 3a. `new_dataset.csv` → `human` + `random`
Load step, then a separate filter step (kept apart so it's clear which rows came from where).

In [ ]:
# Load: read the raw file (or fall back to a synthetic stand-in if it isn't present yet)
if os.path.exists(NEW_DATASET_PATH):
    new_dataset = pd.read_csv(NEW_DATASET_PATH)
    if "SeqID" in new_dataset.columns:
        new_dataset.set_index("SeqID", inplace=True)
    USING_DEMO_NEW_DATASET = False
else:
    warnings.warn(f"'{NEW_DATASET_PATH}' not found - using a synthetic stand-in for 'human'/'random'.")
    new_dataset = _make_demo_new_dataset()
    USING_DEMO_NEW_DATASET = True

print(f"Raw new_dataset: {len(new_dataset)} rows; groups present: {sorted(new_dataset[GROUP_COL].unique())}")


In [ ]:
# Filter: new_dataset.csv already carries an explicit Group column - just keep 'human' and 'random'
# (it may contain other groups we don't want here, e.g. earlier random-control variants).
human_random_df = new_dataset.loc[new_dataset[GROUP_COL].isin(["human", "random"]), [SEQ_COL, GROUP_COL]].copy()
human_random_df.groupby(GROUP_COL).size()


### 3b. `delivered_twist_pool.csv` → `human_truncated`
Load step, then the ID-based filter step described above.

In [ ]:
# Load: read the raw twist pool (or fall back to a synthetic stand-in)
if os.path.exists(TWIST_POOL_PATH):
    twist_pool = pd.read_csv(TWIST_POOL_PATH)
    USING_DEMO_TWIST_POOL = False
else:
    warnings.warn(f"'{TWIST_POOL_PATH}' not found - using a synthetic stand-in for 'human_truncated'.")
    twist_pool = _make_demo_twist_pool()
    USING_DEMO_TWIST_POOL = True

print(f"Raw twist pool: {len(twist_pool)} rows; columns: {list(twist_pool.columns)}")


In [ ]:
# Filter: human_truncated is *hidden* in this pool - it isn't a labelled group, it's every row whose
# transcript_id is an Ensembl transcript ID (starts with 'ENST'). Every other ID prefix in this file
# belongs to a different design and is dropped here, in this dedicated filtering step.
is_human_truncated = twist_pool[TWIST_ID_COL].astype(str).str.startswith(TWIST_ID_PREFIX)

human_truncated_df = twist_pool.loc[is_human_truncated, [TWIST_SEQ_COL]].rename(columns={TWIST_SEQ_COL: SEQ_COL})
human_truncated_df[GROUP_COL] = TWIST_GROUP_NAME

print(f"{is_human_truncated.sum()} / {len(twist_pool)} rows started with '{TWIST_ID_PREFIX}' "
      f"and were kept as '{TWIST_GROUP_NAME}'")
human_truncated_df.head()


### 3c. `NT96` / `NT192` / `NT384` — placeholder
**TODO:** no real source has been specified for these three groups yet. Tell me which file/column/ID
pattern identifies them (the same way `transcript_id` does for `human_truncated` above) and this cell
gets replaced with the real load+filter steps. For now they're synthetic so the rest of the notebook
still runs.

In [ ]:
# Load: read samples_by_library.csv, which labels sequences by a 'library' column
# rather than the shared GROUP_COL name. Fall back to a synthetic stand-in if missing.
if os.path.exists(SAMPLES_BY_LIB_PATH):
    samples_by_lib = pd.read_csv(SAMPLES_BY_LIB_PATH)
    USING_DEMO_NT = False
else:
    warnings.warn(f"'{SAMPLES_BY_LIB_PATH}' not found - using a synthetic stand-in for NT groups.")
    samples_by_lib = _make_demo_samples_by_library()
    USING_DEMO_NT = True

# Filter: keep only the three NT libraries we care about, then rename 'library' -> GROUP_COL
# so the rest of the notebook can treat all sources uniformly.
nt_group_df = (
    samples_by_lib
    .loc[samples_by_lib[NT_LIB_COL].isin(NT_GROUPS), [SEQ_COL, NT_LIB_COL]]
    .rename(columns={NT_LIB_COL: GROUP_COL})
    .copy()
)

print(f"Raw samples_by_library: {len(samples_by_lib)} rows; "
      f"libraries present: {sorted(samples_by_lib[NT_LIB_COL].unique())}")
nt_group_df.groupby(GROUP_COL).size().reindex(NT_GROUPS)


### 3d. Combine all sources into one sequence table

In [ ]:
# Stitch every source together with a uniform index.
# The original IDs (SeqID vs transcript_id vs row number) are incompatible across files,
# so we drop them here and assign a simple sequential index for everything that follows.
seqs = pd.concat([
    nt_group_df[[SEQ_COL, GROUP_COL]],
    human_random_df[[SEQ_COL, GROUP_COL]],
    human_truncated_df[[SEQ_COL, GROUP_COL]],
], ignore_index=True)
seqs.index = [f"seq_{i}" for i in range(len(seqs))]
seqs.index.name = "SeqID"

missing_groups = [g for g in GROUPS if g not in seqs[GROUP_COL].unique()]
if missing_groups:
    print("WARNING - groups not found in data:", missing_groups)

print("new_dataset.csv found:           ", not USING_DEMO_NEW_DATASET)
print("delivered_twist_pool.csv found:  ", not USING_DEMO_TWIST_POOL)
print("samples_by_library.csv found:    ", not USING_DEMO_NT)
seqs.groupby(GROUP_COL).size().reindex(GROUPS)


In [ ]:
# Split into one dataframe per group for convenience in every marker section below.
seqs_by_group = {g: seqs.loc[seqs[GROUP_COL] == g].copy() for g in GROUPS}
for g in GROUPS:
    print(f"{g:>16s}: {len(seqs_by_group[g])} sequences")


## 4. Marker 1 — Nucleotide probabilities

### Step 1 — write the function
`src.rna_analysis.GC_Content` already exists but only returns the combined G+C fraction, not the
individual base rates. We add a small helper for that and reuse `GC_Content` alongside it.

In [ ]:
def nucleotide_probabilities(seqs_df, seq_col=SEQ_COL, bases=constants.RNABASES):
    """Per-sequence mononucleotide probabilities P(A), P(C), P(G), P(U), plus GC content
    (GC content itself comes straight from src.rna_analysis.GC_Content)."""
    records = []
    for seq in seqs_df[seq_col]:
        L = len(seq)
        records.append({base: seq.count(base) / L for base in bases})   # one row of base frequencies per sequence
    probs = pd.DataFrame(records, index=seqs_df.index)
    probs["GC Content"] = seqs_df[seq_col].apply(rna.GC_Content)         # reuse the existing repo function
    return probs


### Step 2 — apply to all the data
Run the function on every group and stack the results into one long dataframe with a `Group` column,
then summarise with mean/std per group.

In [ ]:
nt_prob_frames = []
for g in GROUPS:
    probs = nucleotide_probabilities(seqs_by_group[g])   # per-sequence base probabilities for this group
    probs[GROUP_COL] = g                                  # tag rows with their group for later plotting/grouping
    nt_prob_frames.append(probs)
nt_probs_df = pd.concat(nt_prob_frames)   # one row per sequence, across all six groups

# Mean +/- std per group, in GROUPS order, for a quick numeric overview before plotting
nt_summary = nt_probs_df.groupby(GROUP_COL)[list(constants.RNABASES) + ["GC Content"]].agg(["mean", "std"]).reindex(GROUPS)
nt_summary


### Step 3 — plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: mean per-base probability, one bar group per nucleotide group
mean_by_group = nt_probs_df.groupby(GROUP_COL)[list(constants.RNABASES)].mean().reindex(GROUPS)
mean_by_group.plot(kind="bar", ax=axes[0], color=sns.color_palette("muted", n_colors=4))
axes[0].set_ylabel("Mean nucleotide probability")
axes[0].set_xlabel("Group")
axes[0].set_title("Mean per-base probability by group")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(title="Base")

# Right: full distribution of GC content per group (not just the mean)
sns.violinplot(data=nt_probs_df, x=GROUP_COL, y="GC Content", order=GROUPS,
               hue=GROUP_COL, palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title("GC content distribution by group")
axes[1].set_xlabel("Group")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()


## 5. Marker 2 — uORFs in reading frame

### Step 1 — write the function
Skipped: `src.rna_analysis.uORFs` and `src.rna_analysis.counts_concat_all` already do exactly what's
needed (frame-resolved uORF/ouORF counts and lengths, combined across start codons), so nothing new
to write here.

### Step 2 — apply to all the data
For each group, run `uORFs` once per start codon in `START_CODONS`, then combine those five result
tables with `counts_concat_all` (this mirrors how it's already done in `plots.ipynb`). This can be
slow on large datasets since `uORFs` scans every codon position per sequence per start codon.

In [ ]:
uorf_frames = []
for g in GROUPS:
    sub = seqs_by_group[g]
    per_codon = {codon: rna.uORFs(sub, codon) for codon in START_CODONS}   # one result table per start codon
    combined = rna.counts_concat_all(sub, per_codon["AUG"], per_codon["CUG"],
                                      per_codon["GUG"], per_codon["UUG"], per_codon["ACG"])  # merge all 5
    combined[GROUP_COL] = g
    uorf_frames.append(combined)

uorf_df = pd.concat(uorf_frames)   # one row per sequence, across all six groups

uorf_summary_cols = ["all uORFs", "all ouORFs", "all mean uORF lengths", "all max uORF lengths"]
uorf_summary = uorf_df.groupby(GROUP_COL)[uorf_summary_cols].mean().reindex(GROUPS)
uorf_summary


### Step 3 — plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

sns.boxplot(data=uorf_df, x=GROUP_COL, y="all uORFs", order=GROUPS,
            hue=GROUP_COL, palette=PALETTE, legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Total uORFs per sequence")

sns.boxplot(data=uorf_df, x=GROUP_COL, y="all ouORFs", order=GROUPS,
            hue=GROUP_COL, palette=PALETTE, legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Overlapping uORFs (ouORFs) per sequence")

sns.boxplot(data=uorf_df, x=GROUP_COL, y="all mean uORF lengths", order=GROUPS,
            hue=GROUP_COL, palette=PALETTE, legend=False, ax=axes[1, 0])
axes[1, 0].set_title("Mean uORF length [nt]")

sns.boxplot(data=uorf_df, x=GROUP_COL, y="all max uORF lengths", order=GROUPS,
            hue=GROUP_COL, palette=PALETTE, legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Max uORF length [nt]")

for ax in axes.flat:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("uORF statistics across groups (AUG + CUG + GUG + UUG + ACG)", fontsize=15)
fig.tight_layout()
plt.show()


In [ ]:
# Bonus view: reading-frame specificity of AUG-initiated uORFs specifically (frame1/2/3 relative to the CDS)
frame_cols = ["frame1_uORFs", "frame2_uORFs", "frame3_uORFs"]
aug_frames = []
for g in GROUPS:
    sub = rna.uORFs(seqs_by_group[g], "AUG")
    sub[GROUP_COL] = g
    aug_frames.append(sub)
aug_frame_df = pd.concat(aug_frames)

frame_long = aug_frame_df.melt(id_vars=GROUP_COL, value_vars=frame_cols,
                                var_name="Reading frame", value_name="AUG uORF count")
frame_long["Reading frame"] = frame_long["Reading frame"].str.replace("_uORFs", "", regex=False)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=frame_long, x=GROUP_COL, y="AUG uORF count", hue="Reading frame",
            order=GROUPS, errorbar="sd", ax=ax)
ax.set_title("Mean AUG-initiated uORF count by reading frame")
ax.set_xlabel("Group")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 6. Marker 3 — Secondary structure / minimum free energy (MFE)

### Step 1 — write the function
Skipped: `src.rna_analysis.sliding_window_mfe` already folds sliding 100 nt windows with ViennaRNA
and returns the lowest per-nucleotide MFE found - nothing new to write.

### Step 2 — apply to all the data
Groups are subsampled to `MFE_SAMPLE_SIZE` sequences first since ViennaRNA folding is the slowest
step in this notebook by far.

In [ ]:
mfe_frames = []
if MFE_AVAILABLE:
    for g in GROUPS:
        sub = seqs_by_group[g]
        if MFE_SAMPLE_SIZE is not None and len(sub) > MFE_SAMPLE_SIZE:
            sub = sub.sample(MFE_SAMPLE_SIZE, random_state=RANDOM_SEED)   # cap runtime on large groups
        print(f"Folding group '{g}' ({len(sub)} sequences)...")
        mfes = rna.sliding_window_mfe(sub)
        mfe_frames.append(pd.DataFrame({"sliding window mfe": mfes, GROUP_COL: g}, index=sub.index))
    mfe_df = pd.concat(mfe_frames)
    mfe_summary = mfe_df.groupby(GROUP_COL)["sliding window mfe"].agg(["mean", "std", "count"]).reindex(GROUPS)
else:
    mfe_df = pd.DataFrame(columns=["sliding window mfe", GROUP_COL])
    mfe_summary = pd.DataFrame()
    print("Skipping marker 3 - ViennaRNA is not installed.")

mfe_summary


### Step 3 — plot

In [ ]:
if MFE_AVAILABLE and len(mfe_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.violinplot(data=mfe_df, x=GROUP_COL, y="sliding window mfe", order=GROUPS,
                   hue=GROUP_COL, palette=PALETTE, legend=False, ax=axes[0])
    axes[0].set_title("Sliding-window MFE per nt by group")
    axes[0].set_ylabel("MFE / nt")
    axes[0].set_xlabel("Group")
    axes[0].tick_params(axis="x", rotation=30)

    for g in GROUPS:
        sns.kdeplot(data=mfe_df[mfe_df[GROUP_COL] == g], x="sliding window mfe",
                    label=g, color=PALETTE[g], common_norm=False, ax=axes[1])
    axes[1].set_title("MFE distribution by group")
    axes[1].set_xlabel("MFE / nt")
    axes[1].legend(title="Group", fontsize=9)

    fig.tight_layout()
    plt.show()
else:
    print("No MFE plot - ViennaRNA unavailable or no data.")


## 7. Marker 4 — Markov (dinucleotide transition) probabilities

### Step 1 — write the function
Skipped: `src.KLD_calculation.markov_matrix` (transition matrix) and `src.KLD_calculation.jsd`
(Jensen-Shannon divergence between matrices) already exist - nothing new to write.

### Step 2 — apply to all the data
Build one transition matrix per group, plus the underlying single-nucleotide probabilities for
reference.

In [ ]:
markov_matrices = {}
single_probs = {}
for g in GROUPS:
    p_duplet, p_single, matrix = kldcalc.markov_matrix(seqs_by_group[g], seq_column=SEQ_COL)
    markov_matrices[g] = matrix     # 4x4 transition matrix, rows/cols ordered A,C,G,U
    single_probs[g] = p_single      # P(A), P(C), P(G), P(U) for this group

single_probs_df = pd.DataFrame(single_probs).T.reindex(GROUPS)
single_probs_df.index.name = GROUP_COL
single_probs_df


### Step 3 — plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
bases = list(constants.RNABASES)

for ax, g in zip(axes.flat, GROUPS):
    sns.heatmap(markov_matrices[g], annot=True, fmt=".2f", cmap="viridis",
                xticklabels=bases, yticklabels=bases, vmin=0, vmax=1,
                cbar=False, ax=ax)
    ax.set_title(g)
    ax.set_xlabel("to base")
    ax.set_ylabel("from base")

fig.suptitle("First-order Markov transition matrices by group", fontsize=16)
fig.tight_layout()
plt.show()


In [ ]:
# Pairwise Jensen-Shannon divergence: how different is each group's transition structure
# from every other group's, in a single 6x6 view.
jsd_matrix = pd.DataFrame(index=GROUPS, columns=GROUPS, dtype=float)
for g1 in GROUPS:
    for g2 in GROUPS:
        jsd_matrix.loc[g1, g2] = kldcalc.jsd(markov_matrices[g1], markov_matrices[g2])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(jsd_matrix.astype(float), annot=True, fmt=".3f", cmap="rocket_r", ax=ax)
ax.set_title("Pairwise Jensen-Shannon divergence between group Markov matrices")
fig.tight_layout()
plt.show()


## 8. Marker 5 — 3-mer (trinucleotide) frequencies

Trinucleotide composition captures local sequence context beyond simple base frequencies
(marker 1) and dinucleotide transitions (marker 4, Markov). There are 4³ = 64 possible RNA
3-mers. We count them all in a single O(L) sliding-window pass — one scan per sequence, not
64 separate pattern searches — which is both simpler and faster than any dedicated
string-search algorithm for a pattern of length 3.

### Step 1 — write the function

`count_kmers` tallies every k-mer in one left-to-right pass using a `collections.Counter`.
`kmer_freq_table` applies it to every sequence in a group and returns a per-sequence frequency
DataFrame (counts divided by the number of windows = len(seq) − k + 1).

In [ ]:
from collections import Counter
from itertools import product

# All 64 RNA 3-mers in alphabetical order — fixed column index used everywhere below.
ALL_3MERS = ["".join(p) for p in product(constants.RNABASES, repeat=3)]


def count_kmers(seq, k=3):
    """Count every k-mer in seq with a single sliding-window pass.
    Returns a Counter mapping k-mer string -> raw integer count."""
    return Counter(seq[i : i + k] for i in range(len(seq) - k + 1))


def kmer_freq_table(seqs_df, k=3, seq_col=SEQ_COL, all_kmers=None):
    """Return a DataFrame (n_seqs x n_kmers) of relative k-mer frequencies.

    Each cell is: count(kmer, seq) / (len(seq) - k + 1), i.e. the fraction of
    sliding windows in which that k-mer appears.  A fixed column list (all_kmers)
    ensures every group produces a DataFrame with the same 64 columns even when
    some k-mers are absent from a particular group.
    """
    if all_kmers is None:
        all_kmers = ["".join(p) for p in product(constants.RNABASES, repeat=k)]
    records = []
    for seq in seqs_df[seq_col]:
        n_windows = len(seq) - k + 1
        if n_windows < 1:
            warnings.warn(f"Sequence of length {len(seq)} is shorter than k={k}; row set to zero.")
            records.append({km: 0.0 for km in all_kmers})
            continue
        c = count_kmers(seq, k)
        records.append({km: c.get(km, 0) / n_windows for km in all_kmers})
    return pd.DataFrame(records, index=seqs_df.index, columns=all_kmers)


### Step 2 — apply to all the data

Run `kmer_freq_table` on every group and concatenate into one long DataFrame with a `Group`
column attached. This is the single table used for all downstream plots and statistics.

### Step 3 — plot

**Plot A — mean frequency heatmap:** each cell is the mean relative frequency of one 3-mer
(column) in one group (row). Values are z-scored across groups per k-mer so that the colour
axis is comparable across k-mers that differ widely in absolute frequency.

**Plot B — Kruskal-Wallis per k-mer:** a non-parametric test (no normality assumption)
asking whether the distribution of per-sequence frequencies differs across the six groups.
P-values are Benjamini-Hochberg corrected for 64 simultaneous tests. The bar chart ranks
k-mers by −log₁₀(adjusted p) so the most group-discriminating trinucleotides stand out.

**Plot C — top-10 discriminating k-mers:** mean frequency per group for the 10 k-mers
with the smallest adjusted p-value, making it easy to see *which* groups drive the signal.

In [ ]:
# ── Plot A: z-scored mean frequency heatmap (groups × k-mers) ────────────────
fig, ax = plt.subplots(figsize=(20, 4))

# Z-score each k-mer column across the 6 group means so the colour axis is
# variance-normalised — otherwise high-frequency k-mers dominate the scale.
zscored = (mean_kmer_freq - mean_kmer_freq.mean()) / mean_kmer_freq.std().replace(0, 1)

sns.heatmap(zscored, cmap="RdBu_r", center=0, linewidths=0.3,
            xticklabels=True, yticklabels=True, ax=ax,
            cbar_kws={"label": "z-score of mean frequency"})
ax.set_title("Mean 3-mer frequency by group (z-scored across groups per k-mer)", fontsize=14)
ax.set_xlabel("3-mer")
ax.set_ylabel("Group")
ax.tick_params(axis='x', rotation=90, labelsize=7)
fig.tight_layout()
plt.show()


In [ ]:
# ── Plot B: Kruskal-Wallis + BH correction — which 3-mers differ across groups? ──
from scipy.stats import kruskal as kruskal_wallis
from scipy.stats import false_discovery_control

kw_results = {}
for km in ALL_3MERS:
    # Build one array of per-sequence frequencies per group
    groups_data = [kmer_df.loc[kmer_df[GROUP_COL] == g, km].values for g in GROUPS]
    # KW needs at least 2 groups with non-zero variance; skip degenerate k-mers
    non_trivial = [d for d in groups_data if len(d) > 0 and d.std() > 0]
    if len(non_trivial) < 2:
        kw_results[km] = (np.nan, np.nan)
        continue
    stat, p = kruskal_wallis(*groups_data)
    kw_results[km] = (stat, p)

kw_df = pd.DataFrame(kw_results, index=["H_statistic", "p_value"]).T.dropna()

# Benjamini-Hochberg FDR correction across the 64 simultaneous tests
kw_df["p_adj"] = false_discovery_control(kw_df["p_value"].values, method="bh")
kw_df["neg_log10_p_adj"] = -np.log10(kw_df["p_adj"].clip(lower=1e-300))
kw_df = kw_df.sort_values("neg_log10_p_adj", ascending=False)

ALPHA = 0.05
sig = kw_df["p_adj"] < ALPHA

fig, ax = plt.subplots(figsize=(20, 5))
colors = [PALETTE.get("human", "steelblue") if s else "lightgrey" for s in sig.loc[kw_df.index]]
ax.bar(kw_df.index, kw_df["neg_log10_p_adj"], color=colors, edgecolor="none")
ax.axhline(-np.log10(ALPHA), color="black", linestyle="--", linewidth=1,
           label=f"FDR = {ALPHA} threshold")
ax.set_xlabel("3-mer (ranked by significance)")
ax.set_ylabel(r"$-\log_{10}$(BH-adjusted p-value)")
ax.set_title("Kruskal-Wallis per 3-mer: which trinucleotides differ across the 6 groups?", fontsize=13)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.legend()
fig.tight_layout()
plt.show()

print(f"{sig.sum()} / {len(kw_df)} k-mers significant at FDR < {ALPHA}")
kw_df.head(10)


In [ ]:
# ── Plot C: mean frequency of the top-10 most discriminating k-mers by group ──
top_kmers = kw_df.head(10).index.tolist()

# Melt to long format so seaborn can draw grouped bars with per-sequence error bars
top_long = kmer_df[top_kmers + [GROUP_COL]].melt(id_vars=GROUP_COL,
                                                   var_name="3-mer",
                                                   value_name="Relative frequency")

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=top_long, x="3-mer", y="Relative frequency",
            hue=GROUP_COL, hue_order=GROUPS, palette=PALETTE,
            errorbar="sd", ax=ax)
ax.set_title("Mean frequency (±SD) of top-10 discriminating 3-mers by group")
ax.set_xlabel("3-mer")
ax.tick_params(axis='x', rotation=30)
ax.legend(title="Group", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()


## 8. Cross-marker summary

A compact overview combining one headline statistic per marker for each group.

In [ ]:
summary = pd.DataFrame(index=GROUPS)
summary["GC content (mean)"]      = nt_probs_df.groupby(GROUP_COL)["GC Content"].mean().reindex(GROUPS)
summary["uORFs per seq (mean)"]   = uorf_df.groupby(GROUP_COL)["all uORFs"].mean().reindex(GROUPS)
summary["ouORFs per seq (mean)"]  = uorf_df.groupby(GROUP_COL)["all ouORFs"].mean().reindex(GROUPS)
if MFE_AVAILABLE and len(mfe_df) > 0:
    summary["MFE per nt (mean)"]  = mfe_df.groupby(GROUP_COL)["sliding window mfe"].mean().reindex(GROUPS)
summary["JSD vs human (Markov)"]  = jsd_matrix["human"].reindex(GROUPS)
# Top KW-significant 3-mer: mean frequency per group as a quick marker proxy
if len(kw_df) > 0:
    top1 = kw_df.index[0]
    summary[f"3-mer '{top1}' freq (mean)"] = kmer_df.groupby(GROUP_COL)[top1].mean().reindex(GROUPS)

summary.style.background_gradient(cmap='Blues', axis=0)


---
**Open items:**
- Tell me how `NT96`/`NT192`/`NT384` should actually be sourced (file + column/ID rule, like the
  `transcript_id`/`ENST` rule used for `human_truncated`) and section 3c gets swapped for the real thing.
- Once `data/new_dataset.csv` and `data/delivered_twist_pool.csv` are in place, re-run from section 3 —
  everything downstream already points at `seqs_by_group`, so nothing else needs to change.
- Increase/disable `MFE_SAMPLE_SIZE` once you've confirmed the pipeline behaves as expected on the full data.
